In [1]:
import torch
from typing import Callable,List,Optional,Union
from einops import rearrange

问题（compute_group_normalized_rewards）：组归一化（2分）
交付要求：实现compute_group_normalized_rewards方法，计算每个滚动响应的原始奖励，在组内进行归一化，并返回归一化奖励、原始奖励及有用的元数据。

In [2]:
def compute_group_normalized_rewards(
    reward_fn: Callable[[str, str], dict[str, float]],
    rollout_responses,
    repeated_ground_truths,
    group_size,
    advantage_eps,
    normalize_by_std,
):
    """为每组 rollout 响应计算奖励，并按组进行归一化。

    参数：
    reward_fn: Callable[[str, str], dict[str, float]]  
        用于将 rollout 响应与标准答案（ground truth）进行比较并打分的函数，返回一个字典，
        包含键 "reward"、"format_reward" 和 "answer_reward"。
    
    rollout_responses: list[str]  
        策略生成的 rollout 响应列表。该列表长度为 rollout_batch_size，
        即 rollout_batch_size = n_prompts_per_rollout_batch * group_size。
    
    repeated_ground_truths: list[str]  
        每个样本对应的标准答案列表。该列表长度也为 rollout_batch_size，
        因为每个问题的标准答案被重复了 group_size 次（与每个问题对应的多个响应对齐）。
    
    group_size: int  
        每个问题（即每组）生成的响应数量。
    
    advantage_eps: float  
        用于归一化时避免除零的小常数。
    
    normalize_by_std: bool  
        若为 True，则用每组奖励的标准差进行归一化（即减去均值后除以标准差）；
        否则仅减去组内均值。

    返回：
    tuple[torch.Tensor, torch.Tensor, dict[str, float]]
        - advantages: shape (rollout_batch_size,)，每条 rollout 响应的组内归一化奖励（即优势值）。
        - raw_rewards: shape (rollout_batch_size,)，每条 rollout 响应的原始未归一化奖励。
        - metadata: 用户自定义的其他统计信息，可用于日志记录（例如奖励的均值、标准差、最大/最小值等）。
    """
    batch_size = len(rollout_responses) // group_size

    # 获得每一组的奖励 b*g,
    raw_rewards = [reward_fn(r,g)['reward'] for r,g in zip(rollout_responses,repeated_ground_truths)] # list[float]
    raw_rewards = torch.tensor(raw_rewards) # shape: b*g,

    # 获得优势函数
    rewards = rearrange(raw_rewards,'(b g)->b g',b=batch_size,g=group_size)
    avg_rewards = rewards.mean(dim=-1,keepdim=True)
    if normalize_by_std:
        std_rewards = rewards.std(dim=-1,keepdim=True)
        advantages = (rewards - avg_rewards) / (std_rewards + advantage_eps)
    else:
        advantages = rewards - avg_rewards
    
    advantages  = rearrange(advantages,'b g->(b g)',b=batch_size,g=group_size)# b*g
    metadata = {
        'avg_reward':raw_rewards.mean().item(),
        'std_reward':raw_rewards.std().item(),
        'max_reward':raw_rewards.max().item(),
        'min_reward':raw_rewards.max().item(),
    }
    return advantages, raw_rewards, metadata

In [3]:
def compute_naive_policy_gradient_loss(
    raw_rewards_or_advantages: torch.Tensor,
    policy_log_probs: torch.Tensor,
) -> torch.Tensor:
    """
    计算每个token的策略梯度损失，其中raw_rewards_or_advantages可为原始奖励或已归一化的优势值
    
    参数：
        raw_rewards_or_advantages: 形状为(batch_size, 1)的张量，每个滚动响应的标量奖励/优势值
        policy_log_probs: 形状为(batch_size, sequence_length)的张量，每个token的对数概率
    
    返回：
        形状为(batch_size, sequence_length)的张量，逐token策略梯度损失（将在训练循环中跨批次和序列维度聚合）
    """
    return - policy_log_probs * raw_rewards_or_advantages

In [4]:
def compute_grpo_clip_loss(
    advantages: torch.Tensor,
    policy_log_probs: torch.Tensor,
    old_log_probs: torch.Tensor,
    cliprange: float,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    """    
    参数：
        advantages: 形状为(batch_size, 1)的张量，每个样本的优势值A
        policy_log_probs: 形状为(batch_size, sequence_length)的张量，待训练策略的逐token对数概率
        old_log_probs: 形状为(batch_size, sequence_length)的张量，旧策略的逐token对数概率
        cliprange: 裁剪参数ε（例如0.2）
    
    返回：
        tuple[torch.Tensor, dict[str, torch.Tensor]]:
            loss: 形状为(batch_size, sequence_length)的张量，逐token裁剪损失
            metadata: 需记录的元数据（建议记录每个token是否被裁剪，即min函数右侧的裁剪后损失是否小于左侧）
    """

    # ratio
    ratio = torch.exp(policy_log_probs - old_log_probs) # shape: b l

    # unclipped
    unclipped_part = ratio * advantages # b l

    # clipped 
    clipped_ratio = torch.clamp(ratio, 1.0 - cliprange, 1.0 + cliprange) # b l
    clipped_part = clipped_ratio * advantages # b l

    loss = -torch.min(unclipped_part, clipped_part) # b l

    is_clipped = (clipped_part > unclipped_part).float() # b l
    is_clipped_ratio = is_clipped.sum() / is_clipped.numel()
    metadata = {
        'is_clipped': is_clipped,# b l: float
        'is_clipped_ratio': is_clipped_ratio.item(), # 记录被裁剪的比例
    }
    return loss, metadata

In [5]:
advantages = torch.randn(size=(4,1))
policy_log_probs = torch.randn(size=(4,5))
old_log_probs = torch.randn(size=(4,5))
cliprange = 0.2
loss, metadata = compute_grpo_clip_loss(
    advantages=advantages,
    policy_log_probs=policy_log_probs,
    old_log_probs=old_log_probs,
    cliprange=cliprange,
)
policy_log_probs,old_log_probs,loss, metadata

(tensor([[ 0.4240, -0.7151,  0.3459, -0.1350,  0.1423],
         [-0.5673, -1.1310,  0.2888,  1.5170, -0.6181],
         [-0.7384,  2.4272, -0.1887, -0.7360,  0.9747],
         [ 0.8745,  0.4663,  0.5768, -0.2291, -0.2015]]),
 tensor([[-0.0952,  0.9426,  0.5180,  0.3231,  0.3442],
         [-1.3740,  2.8841, -0.5249,  1.2661,  0.3993],
         [ 1.7394, -0.9986,  0.1491, -1.1243,  0.5551],
         [ 1.0677,  0.8744,  1.6006,  1.8766, -0.8270]]),
 tensor([[ 0.3286,  0.1564,  0.1646,  0.1564,  0.1597],
         [ 1.4617,  0.5219,  1.4720,  0.8384,  0.5219],
         [ 0.2873, 11.0435,  0.2873,  0.5296,  0.5464],
         [ 0.4470,  0.4339,  0.4339,  0.4339,  1.0137]]),
 {'is_clipped': tensor([[1., 0., 0., 0., 0.],
          [1., 0., 1., 1., 0.],
          [0., 1., 0., 1., 1.],
          [0., 0., 0., 0., 1.]]),
  'is_clipped_ratio': 0.4000000059604645})

策略梯度损失包装器
我们将通过对比实验验证三种策略梯度变体：
+ (a) 无基线（no_baseline）：无基线的朴素策略梯度损失，优势值直接为原始奖励A=R(q, o)
+ (b) reinforce_with_baseline：使用组归一化奖励作为优势值（advantage）的朴素策略梯度损失。如果 r ˉ \bar{r} 
r是来自 compute_group_normalized_rewards 的组归一化奖励（可能已或未按组标准差归一化），那么优势值 A = r ˉ A = \bar{r}A= 
（c）grpo_clip：GRPO-Clip 损失函数
+ 为方便起见，我们将实现一个包装器（wrapper），使我们能够轻松在这三种策略梯度损失函数之间切换。


问题（compute_policy_gradient_loss）：策略梯度包装器（1分）
交付要求：实现 compute_policy_gradient_loss 函数——一个便捷包装器，用于调度至对应的损失计算流程（no_baseline、reinforce_with_baseline 或 grpo_clip），并返回逐token损失（per-token loss）及所有辅助统计信息。

In [6]:
from typing import Literal
def compute_policy_gradient_loss(
    policy_log_probs: torch.Tensor,
    loss_type: Literal["no_baseline", "reinforce_with_baseline", "grpo_clip"],
    advantages: torch.Tensor | None = None,
    raw_rewards: torch.Tensor | None = None,
    old_log_probs: torch.Tensor | None = None,
    cliprange: float | None = None,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    if loss_type == "no_baseline":
        assert raw_rewards is not None,"raw_rewards must be provided for no_baseline loss type"
        metadata = {
            'raw_rewards': raw_rewards,
        }
        return compute_naive_policy_gradient_loss(
            raw_rewards_or_advantages=raw_rewards,
            policy_log_probs=policy_log_probs,
        ), metadata

    if loss_type == "reinforce_with_baseline":
        assert advantages is not None,"advantages must be provided for reinforce_with_baseline loss type"
        metadata = {
            'advantages': advantages,
        }
        return compute_naive_policy_gradient_loss(
            raw_rewards_or_advantages=advantages,
            policy_log_probs=policy_log_probs,
        ), metadata
    
    if loss_type == "grpo_clip":
        assert advantages is not None,"advantages must be provided for grpo_clip loss type"
        assert old_log_probs is not None,"old_log_probs must be provided for grpo_clip loss type"
        assert cliprange is not None,"cliprange must be provided for grpo_clip loss type"
        metadata = {
            'cliprange':cliprange,
            'advantages':advantages,
            'old_log_probs':old_log_probs,
        }
        loss,grpo_metadata = compute_grpo_clip_loss(
            advantages=advantages,
            policy_log_probs=policy_log_probs,
            old_log_probs=old_log_probs,
            cliprange=cliprange,
        )
        metadata.update(grpo_metadata)
        return loss,metadata

In [9]:
advantages = torch.randn(size=(4,1))
policy_log_probs = torch.randn(size=(4,5))
old_log_probs = torch.randn(size=(4,5))
cliprange = 0.2
compute_policy_gradient_loss(
    policy_log_probs=policy_log_probs,
    loss_type="grpo_clip",
    advantages=advantages,
    old_log_probs=old_log_probs,
    cliprange=cliprange,
)

(tensor([[ 1.0667,  1.3275,  1.9505, 14.0555,  1.0667],
         [-0.2215, -0.0589, -0.5541, -0.5541, -0.5541],
         [-0.1143, -0.1143, -0.0500, -0.1143, -0.0246],
         [-0.1744, -0.0416, -0.1675, -0.1193, -0.1744]]),
 {'cliprange': 0.2,
  'advantages': tensor([[-1.3334],
          [ 0.4618],
          [ 0.0952],
          [ 0.1453]]),
  'old_log_probs': tensor([[ 2.3249,  0.6069,  0.0507, -0.6211,  0.5089],
          [ 1.1996,  1.2420, -0.5020,  0.4947, -0.1494],
          [ 0.1099,  0.1344,  0.3573, -0.3993,  0.1024],
          [-0.4922,  0.4970,  0.8777,  0.1999, -0.3623]]),
  'is_clipped': tensor([[0., 0., 1., 1., 0.],
          [1., 1., 0., 0., 0.],
          [0., 0., 1., 0., 1.],
          [0., 1., 0., 0., 0.]]),
  'is_clipped_ratio': 0.3499999940395355})

掩码均值（Masked Mean）
截至目前，我们已具备计算优势函数（advantages）、对数概率、逐token损失，以及逐token熵、裁剪比例等辅助统计信息的计算能力。为将形状为（batch_size, sequence_length）的 per-token loss tensors 缩减为损失向量（每个样本对应一个标量损失），我们将在序列维度上计算损失的均值，但仅包含响应对应的索引（即 mask[i, j]==1 的token位置）。

在大多数基于大语言模型（LLM）的强化学习（RL）代码库中，按序列长度归一化是标准操作，但这一做法的合理性尚未明确——观察公式（21）中策略梯度估计的定义可知，其中并不存在归一化因子 1 T ( i ) \frac{1}{T^{(i)}} 
T 
(i)
 
1.我们将先采用这一标准方法（通常称为 masked_mean），后续再测试SFT阶段实现的 masked_normalize 方法。
该函数支持指定均值计算的维度：若 dim = None，则对所有掩码为1的元素计算均值。这一功能可用于获取响应token的平均逐token熵、裁剪比例等统计信息。
问题（masked_mean）：掩码均值（1分）
交付要求：实现 masked_mean 方法，在尊重布尔掩码（boolean mask）的前提下对张量元素求平均。

In [25]:
def masked_mean(
    tensor: torch.Tensor,
    mask: torch.Tensor,# bool
    dim: int | None = None,
) -> torch.Tensor:
    """
    masked_tensor = tensor * mask
    return masked_tensor.mean(dim=dim)
    这个代码是错误的,会影响均值
    """
    valid_elements = mask.float().sum(dim=dim)# b l-> b
    masked_tensor = tensor * mask
    masked_sum = masked_tensor.sum(dim=dim) # b l -> b
    masked_sum[valid_elements == 0.0] = 0.0
    return masked_sum / valid_elements

GRPO 微批次训练步骤（GRPO Microbatch Train Step）
现在我们可以实现 GRPO 的单个microbatch train step（回想一下，对于一个训练小批次，若 gradient_accumulation_steps > 1，我们会迭代多个microbatch）。

具体而言，给定原始奖励（raw rewards）或优势函数（advantages）及对数概率，我们将计算逐token损失，通过 masked_mean 聚合为每个样本的标量损失，在批次维度上求平均，根据梯度累积步数调整损失，并执行反向传播。

问题（grpo_microbatch_train_step）：微批次训练步骤（3分）
交付要求：实现 GRPO 的单个微批次更新，包括策略梯度损失计算、掩码均值聚合及梯度缩放。

In [ ]:
def grpo_microbatch_train_step(
    policy_log_probs: torch.Tensor,
    response_mask: torch.Tensor,
    gradient_accumulation_steps: int,
    loss_type: Literal["no_baseline", "reinforce_with_baseline", "grpo_clip"],
    raw_rewards: torch.Tensor | None = None,
    advantages: torch.Tensor | None = None,
    old_log_probs: torch.Tensor | None = None,
    cliprange: float | None = None,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    loss, metadata = compute_policy_gradient_loss(
        policy_log_probs=policy_log_probs,
        loss_type=loss_type,
        advantages=advantages,
        raw_rewards=raw_rewards,
        old_log_probs=old_log_probs,
        cliprange=cliprange,
    )

    # 这一步与标准的PPO/GRPO实现不同,\sum{i=1}^T log \pi(a_i|s_i)
    masked_loss = masked_mean(
        tensor = loss,
        mask = response_mask,
        dim = -1,
    ).mean() # b -> 1

    # 梯度累积和反向传播
    scaled_loss = masked_loss / gradient_accumulation_steps
    scaled_loss.backward()
    
    metadata.update({
        'scaled_loss': scaled_loss.item(),# 1
        'loss': masked_loss.item(),# 1
    })
    return scaled_loss, metadata

In [27]:
advantages = torch.randn(size=(4,1))
policy_log_probs = torch.randn(size=(4,5))
response_mask = torch.randint(0,1,size=(4,5)).bool()
old_log_probs = torch.randn(size=(4,5))
cliprange = 0.2
grpo_microbatch_train_step(
    policy_log_probs,
    response_mask,
    gradient_accumulation_steps=4,
    loss_type="grpo_clip",
    raw_rewards=None,
    advantages=advantages,
    old_log_probs=old_log_probs,
    cliprange=cliprange,
)

tensor([[ 4.2111,  2.4500,  0.1256,  1.1822,  0.1256],
        [-0.1471, -0.1471, -0.0365, -0.0756, -0.1471],
        [-0.2135, -0.7831, -0.6045, -0.7831, -0.6450],
        [ 0.3237,  0.2010,  0.2010,  0.3214,  0.2760]])


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn